In [4]:
import gymnasium as gym
import numpy as np

class missile_interception_3d(gym.Env):
    def __init__(self):
        # 1. Define Action Space: magnitude + direction on unit circle (cos, sin)
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(3,), dtype=np.float32)
        
        # 2. Define Observation Space (16D ego-frame, no actuator state)
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(16,), dtype=np.float32)

        self.np_random = np.random.RandomState()
        
        # 3. Time Settings
        self.dt_act = 0.1             
        self.n_substeps = 1      
        self.dt_sim = self.dt_act / self.n_substeps 
        self.t_max = 650.0 # REVISAR IF THIS IS ENOF         

        # 4. Physical Limits
        self.a_max = 350.0   # Max G-force (m/s^2) ~35G
        self.g = 9.81        
        self.collision_radius = 300.0

        # Stage-1 curriculum: short range, reduced speed
        self.p_easy = 1.0                   
        self.range_min = 5_000.0           
        self.range_easy_max = 10_000.0     
        self.range_hard_max = 10_000.0   

        self.targetbox_x_min = -15000
        self.targetbox_x_max = 15000
        self.targetbox_y_min = -15000
        self.targetbox_y_max = 15000

        # --- Reward: gated alpha/omega progress ---
        self.R_gate = 5000.0
        self.w_alpha = 2.0
        self.w_omega = 0.5

        # Terminal
        self.r_hit_bonus = 100.0
        self.r_def_ground_pen = 10.0

    def generate_enemy_missile(self):
        if self.np_random.rand() < self.p_easy:
            self.range_max_used = self.range_easy_max
        else:
            self.range_max_used = self.range_hard_max

        range_min = self.range_min
        self.attack_target_x = self.np_random.uniform(self.targetbox_x_min, self.targetbox_x_max)
        self.attack_target_y = self.np_random.uniform(self.targetbox_y_min, self.targetbox_y_max)
        self.enemy_launch_angle = self.np_random.uniform(0, 2 * np.pi)
        self.enemy_theta = self.np_random.uniform(0.523599, 1.0472) 

        self.range_max_used = max(self.range_max_used, range_min + 1.0)
        lower_limit = np.sqrt((range_min * self.g) / np.sin(2 * self.enemy_theta))
        upper_limit = np.sqrt((self.range_max_used * self.g) / np.sin(2 * self.enemy_theta))
        self.enemy_initial_velocity = self.np_random.uniform(lower_limit, upper_limit)
        self.enemy_initial_velocity = float(np.clip(self.enemy_initial_velocity, 700.0, 1100.0))

        ground_range = (
            self.enemy_initial_velocity * np.cos(self.enemy_theta)
            * (2 * self.enemy_initial_velocity * np.sin(self.enemy_theta) / self.g)
        )

        self.enemy_launch_x = self.attack_target_x + ground_range * np.cos(self.enemy_launch_angle)
        self.enemy_launch_y = self.attack_target_y + ground_range * np.sin(self.enemy_launch_angle)
        self.enemy_z = 0
        self.enemy_x = self.enemy_launch_x
        self.enemy_y = self.enemy_launch_y
        self.enemy_pos = np.array([self.enemy_x, self.enemy_y, self.enemy_z], dtype=np.float32)
        self.enemy_azimuth = (self.enemy_launch_angle + np.pi) % (2 * np.pi)

    def generate_defense_missile(self):
        self.defense_launch_x = self.np_random.uniform(self.targetbox_x_min, self.targetbox_x_max)
        self.defense_launch_y = self.np_random.uniform(self.targetbox_y_min, self.targetbox_y_max)

        dx = self.enemy_launch_x - self.defense_launch_x
        dy = self.enemy_launch_y - self.defense_launch_y
        az_nominal = np.arctan2(dy, dx)

        # --- Misalignment: single difficulty band (NO mixture) ---
        az_noise_deg = 30.0
        az_noise = self.np_random.uniform(-np.deg2rad(az_noise_deg), +np.deg2rad(az_noise_deg))
        self.defense_azimuth = az_nominal + az_noise

        self.defense_theta = np.deg2rad(45.0)

        base_velocity = 1200.0
        self.defense_initial_velocity = base_velocity

        self.defense_x = self.defense_launch_x
        self.defense_y = self.defense_launch_y
        self.defense_z = 0.0
        self.defense_pos = np.array([self.defense_x, self.defense_y, self.defense_z], dtype=np.float32)
    
    def _smoothstep(self, x: float) -> float:
        """Smooth ramp 0->1 with zero slope at ends, clamps outside [0,1]"""
        x = float(np.clip(x, 0.0, 1.0))
        return x * x * (3.0 - 2.0 * x)

    def _decode_action(self, a):
        """
        a = [m_raw, c_raw, s_raw] in [-1,1]^3
        Returns: (u_right, u_up) in [-1,1] with sqrt(u_right^2 + u_up^2) <= 1
        """
        a = np.asarray(a, dtype=np.float32)
        m_raw, c_raw, s_raw = float(a[0]), float(a[1]), float(a[2])

        m = 0.5 * (m_raw + 1.0)   # [-1,1] -> [0,1]

        v = np.array([c_raw, s_raw], dtype=np.float32)
        n = float(np.linalg.norm(v))
        if n < 1e-6:
            dir2 = np.array([1.0, 0.0], dtype=np.float32)
        else:
            dir2 = v / n

        u_right = float(m * dir2[0])
        u_up    = float(m * dir2[1])
        return np.array([u_right, u_up], dtype=np.float32)

    def calculate_pronav(self):
        eps = 1e-9

        # Relative geometry (use float64 for stability)
        r = (self.enemy_pos - self.defense_pos).astype(np.float64)
        v = self.defense_vel.astype(np.float64)
        vrel = (self.enemy_vel - self.defense_vel).astype(np.float64)

        R = float(np.linalg.norm(r)) + eps
        V = float(np.linalg.norm(v)) + eps

        rhat = r / R
        vhat = v / V

        # Heading error alpha = angle between velocity direction and LOS direction
        cosang = float(np.clip(np.dot(vhat, rhat), -1.0, 1.0))
        alpha = float(np.arccos(cosang))  # radians

        # LOS angular rate omega (world frame)
        omega = np.cross(r, vrel) / (float(np.dot(r, r)) + eps)
        omega_mag = float(np.linalg.norm(omega))

        # Closing speed (positive => closing)
        vc = -float(np.dot(r, vrel)) / R

        # --- PN term ---
        N = 3.0
        a_pn = N * vc * np.cross(omega, rhat)  # lateral accel in world frame

        # --- Acquisition term (turn-to-LOS) ---
        # Perpendicular component of LOS relative to forward direction
        rhat_perp = rhat - float(np.dot(rhat, vhat)) * vhat
        nperp = float(np.linalg.norm(rhat_perp))

        if nperp < 1e-8:
            a_acq = np.zeros(3, dtype=np.float64)
        else:
            rhat_perp /= nperp  # unit sideways "turn toward LOS" direction

            # Curvature-based magnitude: ~k * V^2 / R, saturate later via a_max
            k_acq = 5.0  # try 3.0–8.0
            a_acq = k_acq * (V * V / R) * rhat_perp

        # --- Blend weight w: 0 => pure PN, 1 => pure acquisition ---

        # Alpha-based weight (dominant)
        alpha_on   = np.deg2rad(20.0)   # start blending earlier
        alpha_full = np.deg2rad(55.0)

        x_alpha = (alpha - alpha_on) / (alpha_full - alpha_on + eps)
        w_alpha = self._smoothstep(x_alpha)

        # Omega-based modifier (only boosts acquisition when PN is sleepy)
        omega_full = 0.00
        omega_on   = 0.05   # <-- key: less brittle than 0.02

        x_omega = (omega_on - omega_mag) / (omega_on - omega_full + eps)
        w_omega = self._smoothstep(x_omega)

        # Robust combine: alpha dominates; omega can't fully shut it off
        w = w_alpha * (0.25 + 0.75 * w_omega)

        # Optional: if not closing, force strong acquisition
        if vc <= 0.0:
            w = max(w, 0.9)

        a_ideal = (1.0 - w) * a_pn + w * a_acq

        # Project into your lateral control basis (right/up) and normalize by a_max
        # Note: Environment now handles gravity compensation internally,
        # so ProNav outputs desired NET lateral accel (same semantics as PPO)
        forward, right, up = self._compute_lateral_basis(self.defense_vel)
        a_right = float(np.dot(a_ideal, right))
        a_up    = float(np.dot(a_ideal, up))

        u_raw = np.array([a_right / self.a_max, a_up / self.a_max], dtype=np.float32)
        self._pronav_u_raw = u_raw.copy()
        u = np.clip(u_raw, -1.0, 1.0)
        return pronav_to_polar_action(float(u[0]), float(u[1]))
    
    def _segment_sphere_intersect(self, r0, r1, r_hit):
        dr = r1 - r0
        dr_norm_sq = float(np.dot(dr, dr))
        if dr_norm_sq < 1e-12:
            return float(np.dot(r0, r0)) <= r_hit * r_hit
        s_star = -float(np.dot(r0, dr)) / dr_norm_sq
        s_star = max(0.0, min(1.0, s_star))
        r_closest = r0 + s_star * dr
        return float(np.dot(r_closest, r_closest)) <= r_hit * r_hit
    
    def _get_obs(self):
        eps = 1e-9

        # World-frame relative state
        r_world = (self.enemy_pos - self.defense_pos).astype(np.float64)
        vrel_world = (self.enemy_vel - self.defense_vel).astype(np.float64)

        # Local basis from defense velocity (world frame unit vectors)
        forward, right, up = self._compute_lateral_basis(self.defense_vel)

        # ===============================
        # 1) Ego-frame (body-frame) r and vrel
        # ===============================
        r_body = np.array([
            float(np.dot(r_world, forward)),
            float(np.dot(r_world, right)),
            float(np.dot(r_world, up)),
        ], dtype=np.float64)

        vrel_body = np.array([
            float(np.dot(vrel_world, forward)),
            float(np.dot(vrel_world, right)),
            float(np.dot(vrel_world, up)),
        ], dtype=np.float64)

        # Normalize r_body / vrel_body
        pos_scale = float(self.range_easy_max)   # 10 km
        vel_scale = 2500.0                        # max vrel ~2300 m/s in Stage-1

        r_body_n = (r_body / (pos_scale + eps)).astype(np.float32)
        vrel_body_n = (vrel_body / (vel_scale + eps)).astype(np.float32)

        # ===============================
        # 2) Scalar helpers
        # ===============================
        dist = float(np.linalg.norm(r_world)) + 1e-6
        v_close = -float(np.dot(r_world, vrel_world)) / dist  # positive when closing

        dist_n = np.float32(np.clip(dist / 50_000.0, 0.0, 4.0))
        vclose_n = np.float32(np.clip(v_close / 2000.0, -2.0, 2.0))
        dist_vclose_feat = np.array([dist_n, vclose_n], dtype=np.float32)

        # Defense own vertical state (keep for ground constraint)
        def_z_n = np.float32(np.clip(self.defense_pos[2] / 15_000.0, -1.0, 2.0))
        def_vz_n = np.float32(np.clip(self.defense_vel[2] / 1200.0, -2.0, 2.0))
        def_state_feat = np.array([def_z_n, def_vz_n], dtype=np.float32)

        # ===============================
        # 4) Keep your geometry features (consistent with ego-frame)
        # ===============================
        dist_body = float(np.linalg.norm(r_body)) + 1e-6

        # LOS lateral projections in body frame
        los_right = float(r_body[1] / dist_body)
        los_up    = float(r_body[2] / dist_body)

        # LOS rate omega in body frame: omega = (r x vrel)/||r||^2
        dist2_body = float(np.dot(r_body, r_body)) + eps
        omega_body = np.cross(r_body, vrel_body) / dist2_body

        omega_right = float(omega_body[1])
        omega_up    = float(omega_body[2])

        omega_scale = 2.0
        omega_right_n = float(np.clip(omega_right / omega_scale, -2.0, 2.0))
        omega_up_n    = float(np.clip(omega_up / omega_scale, -2.0, 2.0))

        geom_feat = np.array([los_right, los_up, omega_right_n, omega_up_n], dtype=np.float32)

        # ===============================
        # 5) NEW: kinematics garnish
        # ===============================
        V_def = float(np.linalg.norm(self.defense_vel))
        V_def_n = np.float32(np.clip(V_def / 1500.0, 0.0, 3.0))
        forward_z = np.float32(float(forward[2]))               # dot(forward, world_up) since world_up=[0,0,1]

        kin_feat = np.array([V_def_n, forward_z], dtype=np.float32)

        # Final obs (16D, no actuator state)
        obs = np.concatenate(
            [r_body_n, vrel_body_n, dist_vclose_feat, def_state_feat, geom_feat, kin_feat],
            axis=0
        ).astype(np.float32)

        return obs

    def _compute_lateral_basis(self, velocity):
        """
        Horizon-stable basis:
          forward = along velocity
          right   = world_up x forward  (horizontal right)
          up      = forward x right     (completes orthonormal frame)
        This keeps 'up' as close to world-up as possible and avoids weird twisting.
        """
        speed = float(np.linalg.norm(velocity))
        if speed < 1.0:
            forward = np.array([1.0, 0.0, 0.0], dtype=np.float32)
        else:
            forward = (velocity / speed).astype(np.float32)

        world_up = np.array([0.0, 0.0, 1.0], dtype=np.float32)

        # right = world_up x forward
        right_raw = np.cross(world_up, forward)
        rnorm = float(np.linalg.norm(right_raw))

        # If forward is near world_up, right_raw ~ 0. Pick a consistent fallback.
        if rnorm < 1e-6:
            # Choose a fixed "north" axis in world XY and build right from that
            # This prevents random spinning when vertical.
            north = np.array([1.0, 0.0, 0.0], dtype=np.float32)
            right_raw = np.cross(north, forward)
            rnorm = float(np.linalg.norm(right_raw))
            if rnorm < 1e-6:
                north = np.array([0.0, 1.0, 0.0], dtype=np.float32)
                right_raw = np.cross(north, forward)
                rnorm = float(np.linalg.norm(right_raw))

        right = (right_raw / (rnorm + 1e-9)).astype(np.float32)

        # up = forward x right (not right x forward)
        up_raw = np.cross(forward, right)
        up = (up_raw / (float(np.linalg.norm(up_raw)) + 1e-9)).astype(np.float32)

        return forward, right, up

    def step(self, action):
        if getattr(self, "done", False):
            return self._get_obs(), 0.0, True, False, {"event": "called_step_after_done", "dist": float(np.linalg.norm(self.enemy_pos - self.defense_pos)), "mag_exec": float(np.nan)}
        
        action = np.clip(action, -1.0, 1.0).astype(np.float32)
        action2 = self._decode_action(action)  # 3D polar -> 2D (right, up), ||.|| <= 1
        mag_exec = float(np.linalg.norm(action2))
        assert mag_exec <= 1.0001, f"_decode_action broke invariant: ||action2|| = {mag_exec}"
        
        # Update episode trackers
        self.ep_max_action_mag = max(self.ep_max_action_mag, mag_exec)

        terminated = False
        truncated = False
        event = "running"
        
        for _ in range(self.n_substeps):
            dt = self.dt_sim
            enemy_pos_old = self.enemy_pos.copy()
            defense_pos_old = self.defense_pos.copy()
            
            forward, right, up = self._compute_lateral_basis(self.defense_vel)
            g_vec = np.array([0.0, 0.0, -self.g], dtype=np.float32)
            a_lat = (action2[0] * self.a_max) * right + (action2[1] * self.a_max) * up
            self.defense_vel += (a_lat + g_vec) * dt
            self.defense_pos += self.defense_vel * dt
            self.defense_x, self.defense_y, self.defense_z = self.defense_pos
            
            # Enemy missile: pure ballistic (gravity only)
            self.enemy_vel += g_vec * dt
            self.enemy_pos += self.enemy_vel * dt
            self.enemy_x, self.enemy_y, self.enemy_z = self.enemy_pos
            self.t += dt
            
            r0 = enemy_pos_old - defense_pos_old
            r1 = self.enemy_pos - self.defense_pos
            if self._segment_sphere_intersect(r0, r1, self.collision_radius):
                self.success = True
                terminated = True
                self.done = True
                event = "hit"
                self.time_to_hit = float(self.t)
                self.terminal_event = "hit"
                break
            
            # --- Only real physics terminals ---
            if self.defense_pos[2] < 0.0:
                terminated = True
                self.done = True
                event = "defense_ground"
                self.terminal_event = event
                break

            if self.enemy_pos[2] < 0.0:
                terminated = True
                self.done = True
                event = "enemy_ground"
                self.terminal_event = event
                break

            # --- Debug safety fuse (not a "task" terminal) ---
            if self.t >= self.t_max:
                truncated = True
                self.done = True
                event = "debug_cap"
                self.terminal_event = event
                break

        obs = self._get_obs()
        dist_after = float(np.linalg.norm(self.enemy_pos - self.defense_pos))
        self.min_dist = min(getattr(self, "min_dist", float("inf")), dist_after)
        self.ep_min_dist = min(self.ep_min_dist, float(dist_after))

        # --- Geometry for reward ---
        r = (self.enemy_pos - self.defense_pos).astype(np.float64)
        vrel = (self.enemy_vel - self.defense_vel).astype(np.float64)
        d = float(np.linalg.norm(r)) + 1e-9
        rhat = r / d

        v_def = self.defense_vel.astype(np.float64)
        V_def = float(np.linalg.norm(v_def)) + 1e-9
        vhat = v_def / V_def

        cos_alpha = float(np.clip(np.dot(vhat, rhat), -1.0, 1.0))
        alpha = float(np.arccos(cos_alpha))

        omega_vec = np.cross(r, vrel) / (float(np.dot(r, r)) + 1e-9)
        omega = float(np.linalg.norm(omega_vec))

        R = d

        # --- Gated progress reward ---
        gate = 1.0 / (1.0 + (R / self.R_gate) ** 2)

        if self.prev_alpha is None:
            d_alpha = 0.0
            d_omega = 0.0
        else:
            d_alpha = self.prev_alpha - alpha
            d_omega = self.prev_omega - omega

        self.prev_alpha = alpha
        self.prev_omega = omega

        r_dense = gate * (self.w_alpha * d_alpha + self.w_omega * d_omega)

        terminal_term = 0.0
        if event == "hit":
            terminal_term = self.r_hit_bonus
        elif event == "defense_ground":
            terminal_term = -self.r_def_ground_pen

        reward = float(r_dense) + terminal_term

        if terminated or truncated:
            self.terminal_event = event

        closing = -float(np.dot(rhat, vrel))
        fwd_z = float(self._compute_lateral_basis(self.defense_vel)[0][2])

        info = {
            "event": event,
            "t": float(self.t),
            "dist": float(dist_after),
            "closing": float(closing),
            "alpha_rad": float(alpha),
            "alpha_deg": float(np.rad2deg(alpha)),
            "omega_mag": float(omega),
            "gate": float(gate),
            "d_alpha": float(d_alpha),
            "d_omega": float(d_omega),
            "r_dense": float(r_dense),
            "r_terminal": float(terminal_term),
            "mag_exec": float(mag_exec),
            "defense_z": float(self.defense_pos[2]),
            "defense_vz": float(self.defense_vel[2]),
            "defense_speed": float(V_def),
            "forward_z": float(fwd_z),
        }
        if terminated or truncated:
            info["min_dist"] = float(self.ep_min_dist)
            info["max_action_mag"] = float(self.ep_max_action_mag)
            info["time_to_hit"] = float(self.time_to_hit) if self.time_to_hit is not None else np.nan
        return obs, reward, terminated, truncated, info

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None: self.np_random = np.random.RandomState(seed)
        self.done = False
        self.success = False
        self.t = 0.0
        self.generate_enemy_missile()
        self.generate_defense_missile()
        
        self.defense_vel = np.array([
            self.defense_initial_velocity * np.cos(self.defense_azimuth) * np.cos(self.defense_theta),
            self.defense_initial_velocity * np.sin(self.defense_azimuth) * np.cos(self.defense_theta),
            self.defense_initial_velocity * np.sin(self.defense_theta)
        ], dtype=np.float32)
        
        self.enemy_vel = np.array([
            self.enemy_initial_velocity * np.cos(self.enemy_azimuth) * np.cos(self.enemy_theta),
            self.enemy_initial_velocity * np.sin(self.enemy_azimuth) * np.cos(self.enemy_theta),
            self.enemy_initial_velocity * np.sin(self.enemy_theta)
        ], dtype=np.float32)
        
        self.defense_pos = np.array([self.defense_x, self.defense_y, self.defense_z], dtype=np.float32)
        self.enemy_pos = np.array([self.enemy_x, self.enemy_y, self.enemy_z], dtype=np.float32)
        d0 = float(np.linalg.norm(self.enemy_pos - self.defense_pos))
        Vd = float(np.linalg.norm(self.defense_vel)) + 1e-6
        self.t_max = min(60.0, 3.0 * d0 / Vd)
        self.min_dist = d0
        self.ep_min_dist = d0
        self.ep_max_action_mag = 0.0
        self.time_to_hit = None
        self.terminal_event = "running"
        self.prev_alpha = None
        self.prev_omega = None
        
        return self._get_obs(), {}


def pronav_to_polar_action(u_right, u_up):
    """Convert old-style 2D (right, up) action to new 3D polar (m_raw, c_raw, s_raw)."""
    u = np.array([u_right, u_up], dtype=np.float32)
    m = float(np.linalg.norm(u))
    if m < 1e-6:
        return np.array([-1.0, 1.0, 0.0], dtype=np.float32)  # magnitude 0, arbitrary direction
    dir2 = u / m
    m = min(m, 1.0)
    m_raw = 2.0 * m - 1.0  # [0,1] -> [-1,1]
    return np.array([m_raw, float(dir2[0]), float(dir2[1])], dtype=np.float32)


# ==========================================
# EVALUATION FUNCTION (separate from training reward)
# ==========================================
def evaluate_policy(env, policy_fn, n_episodes=100, seed0=0):
    """
    Evaluate a policy and return episode-level metrics.
    This is separate from training reward - these metrics are what you actually care about.
    
    Args:
        env: missile_interception_3d environment instance
        policy_fn: Function that takes obs and returns action
        n_episodes: Number of episodes to evaluate
        seed0: Starting seed (episodes use seed0, seed0+1, ..., seed0+n_episodes-1)
    
    Returns:
        summary: Dict with aggregated metrics (hit_rate, min_dist stats, etc.)
        metrics: Dict with raw episode data
    """
    metrics = {
        "hits": 0,
        "ground_defense": 0,
        "ground_enemy": 0,
        "diverged": 0,
        "timeout": 0,
        "min_dist_list": [],
        "time_to_hit_list": [],
        "max_g_list": [],
    }

    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + i)
        done = False

        while not done:
            action = policy_fn(obs)  # your PPO policy OR env.calculate_pronav() baseline
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated

        event = info["event"]
        metrics["min_dist_list"].append(env.ep_min_dist)
        metrics["max_g_list"].append(env.ep_max_action_mag)

        if event == "hit":
            metrics["hits"] += 1
            if env.time_to_hit is not None:
                metrics["time_to_hit_list"].append(env.time_to_hit)
        elif event == "defense_ground":
            metrics["ground_defense"] += 1
        elif event == "enemy_ground":
            metrics["ground_enemy"] += 1
        elif event == "diverged":
            metrics["diverged"] += 1
        elif event == "timeout":
            metrics["timeout"] += 1

    hit_rate = metrics["hits"] / n_episodes

    summary = {
        "hit_rate": hit_rate,
        "min_dist_mean": float(np.mean(metrics["min_dist_list"])),
        "min_dist_p50": float(np.median(metrics["min_dist_list"])),
        "min_dist_p10": float(np.percentile(metrics["min_dist_list"], 10)),
        "time_to_hit_mean": float(np.mean(metrics["time_to_hit_list"])) if metrics["time_to_hit_list"] else None,
        "max_g_mean": float(np.mean(metrics["max_g_list"])),
        "violations": {
            "defense_ground": metrics["ground_defense"],
            "enemy_ground": metrics["ground_enemy"],
            "diverged": metrics["diverged"],
            "timeout": metrics["timeout"],
        }
    }
    return summary, metrics

def run_baseline():
    env = missile_interception_3d()
    outcomes = []
    min_distances = []
    ep_thrust_avgs = []
    ep_raw_maxes = []

    N_EPISODES = 50
    print(f"Running {N_EPISODES} episodes of Augmented ProNav...\n")

    for i in range(N_EPISODES):
        obs, _ = env.reset(seed=i)
        done = False
        ep_mag_exec = []
        ep_u_raw_mag = []

        while not done:
            action = env.calculate_pronav()
            u_raw = env._pronav_u_raw.copy()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            ep_mag_exec.append(info['mag_exec'])
            ep_u_raw_mag.append(float(np.linalg.norm(u_raw)))

        outcomes.append(info['event'])
        min_distances.append(env.ep_min_dist)
        avg_thrust = np.mean(ep_mag_exec)
        max_raw = np.max(ep_u_raw_mag)
        avg_raw = np.mean(ep_u_raw_mag)
        ep_thrust_avgs.append(avg_thrust)
        ep_raw_maxes.append(max_raw)

        print(f"Ep {i+1:02d} | {info['event']:<14} | minD {env.ep_min_dist:6.1f}m | thrust avg {avg_thrust*100:5.1f}% | u_raw avg {avg_raw*100:5.1f}% max {max_raw*100:5.1f}%")

    hits = outcomes.count("hit")
    non_hit = [d for d, e in zip(min_distances, outcomes) if e != "hit"]
    print(f"\n--- SUMMARY ({N_EPISODES} eps) ---")
    print(f"Hit Rate:        {hits}/{N_EPISODES} ({hits/N_EPISODES*100:.1f}%)")
    print(f"Miss Dist (non-hit): {np.mean(non_hit):.1f} m" if non_hit else "Miss Dist:       N/A (all hits)")
    print(f"Avg Thrust:      {np.mean(ep_thrust_avgs)*100:.1f}%  (decoded ||action2||)")
    print(f"Max u_raw seen:  {np.max(ep_raw_maxes)*100:.1f}%  (pre-clip ProNav demand)")

# if __name__ == "__main__":
#     run_baseline()

In [5]:
import numpy as np
import gymnasium as gym

class RunningMeanStd:
    """
    Numerically stable running mean/var using Welford's algorithm (vectorized).
    """
    def __init__(self, shape, eps=1e-4):
        self.mean = np.zeros(shape, dtype=np.float64)
        self.var  = np.ones(shape, dtype=np.float64)
        self.count = float(eps)  # prevents div-by-zero early

    def update(self, x: np.ndarray):
        x = np.asarray(x, dtype=np.float64)
        if x.ndim == 1:
            x = x[None, :]  # (1, D)
        batch_mean = x.mean(axis=0)
        batch_var  = x.var(axis=0)
        batch_count = x.shape[0]

        # merge two sets of stats: current (mean,var,count) and batch (batch_mean,batch_var,batch_count)
        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * (batch_count / tot_count)

        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + (delta * delta) * (self.count * batch_count / tot_count)
        new_var = M2 / tot_count

        self.mean = new_mean
        self.var = new_var
        self.count = float(tot_count)

class NormalizeObsWrapper(gym.Wrapper):
    """
    Online observation normalization:
      obs_norm = clip( (obs - mean)/sqrt(var+eps), -clip, +clip )

    Safety features:
      - can freeze updates (update_stats=False)
      - can bypass normalization entirely (enabled=False)
      - only touches observations; forwards reward/done/info untouched
    """
    def __init__(self, env, eps=1e-8, clip=5.0, enabled=True, update_stats=True):
        super().__init__(env)
        self.eps = float(eps)
        self.clip = float(clip)
        self.enabled = bool(enabled)
        self.update_stats = bool(update_stats)

        obs_shape = env.observation_space.shape
        self.rms = RunningMeanStd(obs_shape)

        # normalized obs is (approximately) unbounded, but we'll keep the same dtype/shape
        self.observation_space = gym.spaces.Box(
            low=-np.inf, high=np.inf, shape=obs_shape, dtype=np.float32
        )

    def _normalize(self, obs: np.ndarray) -> np.ndarray:
        if not self.enabled:
            return obs.astype(np.float32)

        mean = self.rms.mean
        var = self.rms.var
        obs_n = (obs.astype(np.float64) - mean) / np.sqrt(var + self.eps)
        obs_n = np.clip(obs_n, -self.clip, +self.clip)
        return obs_n.astype(np.float32)

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        if self.update_stats:
            self.rms.update(obs)
        return self._normalize(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        if self.update_stats:
            self.rms.update(obs)
        return self._normalize(obs), reward, terminated, truncated, info

    # Convenience controls
    def freeze(self):
        self.update_stats = False

    def unfreeze(self):
        self.update_stats = True

    def set_enabled(self, enabled: bool):
        self.enabled = bool(enabled)

def eval_pronav(env, n=50, seed0=0):
    outcomes = []
    times = []
    mins = []
    for i in range(n):
        obs, _ = env.reset(seed=seed0+i)
        done = False
        while not done:
            a = env.unwrapped.calculate_pronav()  # works even when wrapped
            obs, r, term, trunc, info = env.step(a)
            done = term or trunc
        outcomes.append(info["event"])
        mins.append(env.unwrapped.ep_min_dist)
        times.append(env.unwrapped.time_to_hit if env.unwrapped.time_to_hit is not None else np.nan)
    return outcomes, np.nanmean(times), float(np.mean(mins))

# Base
env0 = missile_interception_3d()
out0, t0, md0 = eval_pronav(env0, n=50, seed0=0)

# Wrapped
env1 = NormalizeObsWrapper(missile_interception_3d(), enabled=True, update_stats=True)
out1, t1, md1 = eval_pronav(env1, n=50, seed0=0)

print("Same outcomes?", out0 == out1)
print("time_to_hit mean:", t0, t1)
print("min_dist mean:", md0, md1)

env = NormalizeObsWrapper(missile_interception_3d(), clip=1e9)
obs, _ = env.reset(seed=0)
# reconstruct
rec = obs.astype(np.float64) * np.sqrt(env.rms.var + env.eps) + env.rms.mean
# Compare against raw obs from unwrapped reset with same seed
env_raw = missile_interception_3d()
obs_raw, _ = env_raw.reset(seed=0)
print("reconstruction max abs error:", float(np.max(np.abs(rec - obs_raw))))

Same outcomes? True
time_to_hit mean: 30.542000000000165 30.542000000000165
min_dist mean: 219.36484008789063 219.36484008789063
reconstruction max abs error: 6.857181489294817e-12
